# Retail Demand Forecasting and Inventory Analytics

Executed review of real pipeline outputs. Run `python src/pipeline.py --source-dir YOUR_FOLDER` and `python src/reporting.py` first. All modeling decisions and limits are documented in the README.

In [1]:
from pathlib import Path
import json, pandas as pd
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
EX=ROOT/'data/exports'
status=json.loads((ROOT/'reports/data_status.json').read_text())
print(json.dumps(status,indent=2))

{
  "source": "Favorita Store Sales competition",
  "holiday_records": 350,
  "missing_files": [],
  "status": "Analysis executed",
  "forecast_results_available": true,
  "future_forecast_rows": 528,
  "sales_rows": 3000888,
  "sales_total": 1073644952.2030684,
  "stores": 54,
  "families": 33,
  "observed_days": 1684,
  "date_min": "2013-01-01",
  "date_max": "2017-08-15",
  "missing_calendar_dates": [
    "2013-12-25",
    "2014-12-25",
    "2015-12-25",
    "2016-12-25"
  ],
  "model_window_start": "2016-12-26",
  "model_window_days": 233,
  "best_model_selected_on_folds_1_2": "Random Forest",
  "final_holdout": {
    "fold": 3,
    "purpose": "Final holdout",
    "method": "Random Forest",
    "MacroFamilyWAPE": 0.22118821037345593,
    "MAE": 2415.8221600489587,
    "RMSE": 7521.971653369633,
    "WAPE": 0.09576821706869583
  },
  "holdout_days": 16,
  "inventory_assumptions": {
    "lead_days": 7,
    "review_days": 7,
    "z_values": [
      0,
      1.28,
      1.65
    ],
   

In [2]:
scores=pd.read_csv(EX/'forecast_metrics.csv')
print(scores.to_string(index=False))
print('Selection folds only:',scores.query('fold<3').groupby('method').MacroFamilyWAPE.mean().idxmin())

 fold         purpose         method  MacroFamilyWAPE         MAE         RMSE     WAPE
    1 Model selection Seasonal naive         0.150860 2610.312304  8911.407941 0.099709
    1 Model selection Moving average         0.253491 4946.168240 14566.372187 0.188934
    1 Model selection  Random Forest         0.141299 1588.516658  4429.046624 0.060678
    1 Model selection        XGBoost         7.212891 1947.595706  5745.145718 0.074394
    2 Model selection Seasonal naive         0.167627 1544.957243  4268.080624 0.059387
    2 Model selection Moving average         0.250638 4191.103589 11822.075749 0.161102
    2 Model selection  Random Forest         0.162522 1540.431421  4210.896368 0.059213
    2 Model selection        XGBoost         8.618953 1792.407085  4839.679678 0.068898
    3   Final holdout Seasonal naive         0.226282 3233.157796 10659.195043 0.128169
    3   Final holdout Moving average         0.260731 3361.600096  8929.203993 0.133261
    3   Final holdout  Random Fo

In [3]:
folds=pd.read_csv(EX/'fold_dates.csv')
print(folds.to_string(index=False))
print('Missing calendar dates:',status['missing_calendar_dates'])

 fold train_start  train_end evaluation_start evaluation_end         purpose
    1  2016-12-26 2017-06-28       2017-06-29     2017-07-14 Model selection
    2  2016-12-26 2017-07-14       2017-07-15     2017-07-30 Model selection
    3  2016-12-26 2017-07-30       2017-07-31     2017-08-15   Final holdout
Missing calendar dates: ['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25']


In [4]:
family=pd.read_csv(EX/'family_forecast_metrics.csv')
winner=status['best_model_selected_on_folds_1_2']
print(family.query('fold==3 and method==@winner').sort_values('WAPE',ascending=False).head(10).to_string(index=False))

 fold        method                     family          MAE         RMSE     WAPE
    3 Random Forest                      BOOKS     1.365347     1.531522 2.427284
    3 Random Forest SCHOOL AND OFFICE SUPPLIES  2353.233480  2510.106972 0.726938
    3 Random Forest            HOME APPLIANCES     6.046950     7.011285 0.559256
    3 Random Forest                  BABY CARE     3.655336     4.449579 0.365534
    3 Random Forest                   CLEANING 14472.310783 16419.951274 0.217375
    3 Random Forest                 LADIESWEAR   112.265040   137.424461 0.206204
    3 Random Forest                   LINGERIE    79.666847    89.775089 0.189796
    3 Random Forest                 GROCERY II   294.083841   336.691606 0.187981
    3 Random Forest                  HOME CARE  2589.339309  2965.309432 0.164472
    3 Random Forest           LIQUOR,WINE,BEER   857.414626  1143.729012 0.163140


In [5]:
inventory=pd.read_csv(EX/'inventory_scenarios.csv')
print(inventory.to_string(index=False))
future=pd.read_csv(EX/'future_forecast.csv')
print('Future rows:',len(future),'Dates:',future.date.min(),future.date.max())

 service_z       demand    fulfilled  lost_sales_units  mean_ending_stock  holding_cost_assumed  fill_rate
      0.00 1.331918e+07 1.328552e+07      33662.681670      183568.254348          9.692404e+05   0.997473
      1.28 1.331918e+07 1.328753e+07      31648.679003      200253.303053          1.057337e+06   0.997624
      1.65 1.331918e+07 1.328809e+07      31091.144499      205077.141407          1.082807e+06   0.997666
Future rows: 528 Dates: 2017-08-16 2017-08-31


## Interpretation

Pooled WAPE is weighted by recorded quantities. Macro-family WAPE makes smaller families visible; neither treats kilograms and item counts as economically equivalent. Promotion patterns are associations. Future accuracy is unavailable until actual sales exist. The inventory exercise is a scenario with stated assumptions, not a replenishment recommendation.